# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()
df_original = df

### TODO 1 — drop duplicates

In [3]:
removed = df.duplicated().sum()         #creates a variable showing number of duoplicated in df
clean = df.drop_duplicates().copy()     #drops duplicates in df
assert (clean.duplicated().sum() == 0)  #checks for no duplicates in df
log('Step 1: drop duplicates', 'dropped duplicated rows', removed)

[Step 1: drop duplicates] dropped duplicated rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
clean['price'] = pd.to_numeric(clean['price'].str.strip().str.replace('$','',regex=False))  #makes all prices floats, removes spaces and $

assert clean['price'].dtype == float    #checks proper formatting for price
log('Step 2: clean price', 'stripped $ and white space, converted all to float\n', clean['price'].count())

[Step 2: clean price] stripped $ and white space, converted all to float
 (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
clean['qty'] = pd.to_numeric(clean['qty'],errors='coerce')  #changes quantities to float
missing = clean['qty'].isna().sum()           #creates variable counting na qty
clean = clean.dropna(subset=['qty']).copy()   #drops na qty rows from df
negative = (clean['qty'] < 0).sum()           #creates variable counting negative qty
clean = clean[clean['qty'] > 0].copy()        #removes negative qty rows from df
assert (clean['qty'] < 0).sum() == 0
log('Step 3: qty -> numeric, drop missing/negative', 'dropped na', missing)
log('Step 3: qty -> numeric, drop missing/negative', 'dropped negatives', negative)

[Step 3: qty -> numeric, drop missing/negative] dropped na (12 row(s))
[Step 3: qty -> numeric, drop missing/negative] dropped negatives (13 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(clean['item'].value_counts())   #prints list of item names and how many times they appear
ITEM_MAP = {'Foam Finger':'Foam Finger','foam finger':'Foam Finger',          #map of item names that removes redundant variations
            'cheese burger':'Cheese Burger','Cheeseburger':'Cheese Burger',
            'Rain Poncho':'Rain Poncho','rain poncho':'Rain Poncho'}
clean['item'] = clean['item'].replace(ITEM_MAP)                               #set item names to map
log('Step 4: canonicalize item', 'built and applied a map removing variations', clean['item'].value_counts().count())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[Step 4: canonicalize item] built and applied a map removing variations (3 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(clean['category'].value_counts())
CATEGORY_MAP = {'Merch':'Merch','Apparel':'Merch','Food':'Food','food':'Food','rain-gear':'Rain Gear','RainGear':'Rain Gear'} #creates map that eliminates redundant category names
clean['category'] = clean['category'].replace(CATEGORY_MAP)     #change df category names to map
log('Step 5: normalize category', 'built and applied a map to remove variation in titles', clean['category'].value_counts().count())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[Step 5: normalize category] built and applied a map to remove variation in titles (3 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
df = clean
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].value_counts().count() == 3       #checks that the proper number of item names are used
assert df['category'].value_counts().count() == 3   #checks that the proper number of category names are used
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']                                 #adds revenue column to df
log('Step 7: the number you would report', 'added revenue column', len(df['revenue']))
print(df[['category','revenue']].groupby('category').sum().round(2))    #prints total revenue by category
print('Revenue Sum:', f'${df['revenue'].sum().round(2)}')               #prints total revenue

[Step 7: the number you would report] added revenue column (275 row(s))
           revenue
category          
Food        1656.0
Merch       1572.0
Rain Gear   1512.0
Revenue Sum: $4740.0


**What I would tell the vendor:** Stock more food because it has the highest revenue.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,Step 1: drop duplicates,dropped duplicated rows,15
1,Step 2: clean price,"stripped $ and white space, converted all to f...",300
2,"Step 3: qty -> numeric, drop missing/negative",dropped na,12
3,"Step 3: qty -> numeric, drop missing/negative",dropped negatives,13
4,Step 4: canonicalize item,built and applied a map removing variations,3
5,Step 5: normalize category,built and applied a map to remove variation in...,3
6,Step 7: the number you would report,added revenue column,275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

In [11]:
df_original['price'] = pd.to_numeric(df_original['price'].str.strip().str.replace('$','',regex=False))    #cleans price
df_original['revenue'] = df_original['price']*df_original['qty']            #create revenue column in copy (df_original)
print('Initial total revenue:', df_original['revenue'].sum())

df_original = df_original.drop_duplicates().copy()                          #drops duplicates in copy
print('Total revenue without duplicates:', df_original['revenue'].sum())

df_original = df_original[df_original['qty']>0].copy()                      #drops negative qty in copy
print('Total revenue without negative', df_original['revenue'].sum())

Initial total revenue: 4852.5
Total revenue without duplicates: 4594.5
Total revenue without negative 4740.0


a) Step 1: removing duplicates, changed my total revenue the most. Before this step, the total revenue was $4,852.50 and after $$4594.50. This is a difference of $$303.

b) I chose to remove orders with negative qty, aka. refunds. Instead, I could have kept refunds, which would have resulted in a total revenue of $$4,594.50 (this is a $145.50 decrease). I did this because it causes the dataframe to show only sales.

b.1) I removed refunds because I was instructed to. It was the most impactful choice, but a choice **I** made was how to format the categories. This had no impact on revenue. Alternatively, one could format it with no capitals or spaces. I didn't do that because I think it looks more official with capitals and spaces.